# 🌾 Krishi-Veda: Fine-tune SmolLM2-135M on Vedic Agricultural Knowledge

Fine-tune HuggingFace's SmolLM2-135M on 150 Assamese/English Vedic farming Q&A pairs.
Output: ~100MB GGUF model for offline ARM64 Android via llama.cpp.

**By Joydeep Das, Silchar, Assam**

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets accelerate peft trl bitsandbytes huggingface_hub sentencepiece protobuf

In [ ]:
# 2. Load training data from GitHub
import json, requests
url = 'https://raw.githubusercontent.com/divineearthly/Krishi-Veda-Module/main/training_data/vedic_farming_smollm2.json'
training_data = requests.get(url).json()
print(f'Loaded {len(training_data)} Vedic farming Q&A pairs')

In [ ]:
# 3. Format for ChatML
from datasets import Dataset

def format_chatml(example):
    msgs = example['messages']
    text = ''
    for m in msgs:
        if m['role'] == 'user':
            text += f"<|im_start|>user\n{m['content']}<|im_end|>\n"
        else:
            text += f"<|im_start|>assistant\n{m['content']}<|im_end|>\n"
    return {'text': text}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_chatml)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f'Train: {len(dataset["train"])}, Test: {len(dataset["test"])}')

In [ ]:
# 4. Load SmolLM2-135M with 4-bit quantization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

model_id = 'HuggingFaceTB/SmolLM2-135M-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
)
print('Model loaded!')

In [ ]:
# 5. Apply LoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 6. Train!
training_args = TrainingArguments(
    output_dir='./krishi-veda-smollm2',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    tokenizer=tokenizer,
    max_seq_length=512,
    dataset_text_field='text',
)

trainer.train()
print('Training complete!')

In [ ]:
# 7. Test the trained model
from transformers import pipeline
pipe = pipeline('text-generation', model=model, tokenizer=tokenizer)
result = pipe("<|im_start|>user\nWhat is Panchgavya?<|im_end|>\n<|im_start|>assistant\n", max_new_tokens=100)
print(result[0]['generated_text'])

In [ ]:
# 8. Merge and save
merged = model.merge_and_unload()
merged.save_pretrained('./krishi-veda-merged')
tokenizer.save_pretrained('./krishi-veda-merged')
print('Model saved! Download the krishi-veda-merged folder.')

## 9. Convert to GGUF on your Android phone

```bash
# On your Termux phone:
python3 ~/llama.cpp/convert_hf_to_gguf.py ./krishi-veda-merged \
    --outfile vedic-krishi-135m.gguf --outtype q4_k_m

# Run it:
./llama-simple -m vedic-krishi-135m.gguf -p "What crop grows in Silchar?"
```